In [3]:
import pandas as pd
import numpy as np

In [9]:
df = pd.read_csv('VehicleInsuranceData.csv')

print("--- Raw Data Shape & Info ---")
print(df.info())


--- Raw Data Shape & Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8630 entries, 0 to 8629
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     8630 non-null   int64  
 1   clv                            8630 non-null   float64
 2   Response                       8630 non-null   object 
 3   Coverage                       8630 non-null   object 
 4   Education                      8630 non-null   object 
 5   EmploymentStatus               8630 non-null   object 
 6   Gender                         8630 non-null   object 
 7   Income                         8630 non-null   int64  
 8   Location.Code                  8630 non-null   object 
 9   Marital.Status                 8630 non-null   object 
 10  Monthly.Premium.Auto           8630 non-null   int64  
 11  Months.Since.Last.Claim        8630 non-null   int64  
 12  Months.Since.Polic

In [10]:
# 2. Clean Column Names (Replace dots with underscores and make lowercase)
df.columns = df.columns.str.replace('.', '_', regex=False).str.lower()

In [11]:
# 3. Handle Data Types & Consistency
# Strip any accidental whitespace from string columns
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].str.strip()

In [12]:
# Ensure numeric columns are explicitly typed
df['clv'] = df['clv'].astype(float)
df['income'] = df['income'].astype(int)
df['monthly_premium_auto'] = df['monthly_premium_auto'].astype(int)
df['total_claim_amount'] = df['total_claim_amount'].astype(float)

In [13]:
# 4. Handle Missing Values / Sanity Check
# Let's see if there are actual missing values
missing_summary = df.isnull().sum()
print("\n--- Missing Values Per Column ---")
print(missing_summary[missing_summary > 0])


--- Missing Values Per Column ---
Series([], dtype: int64)


In [14]:
# 5. Preview the cleaned data
print("\n--- Cleaned Data Preview ---")
print(df.head())


--- Cleaned Data Preview ---
   unnamed: 0           clv response  coverage education employmentstatus  \
0           1   2763.519279       No     Basic  Bachelor         Employed   
1           2   6979.535903       No  Extended  Bachelor       Unemployed   
2           3  12887.431650       No   Premium  Bachelor         Employed   
3           4   7645.861827       No     Basic  Bachelor       Unemployed   
4           5   2813.692575       No     Basic  Bachelor         Employed   

  gender  income location_code marital_status  ...  \
0      F   56274      Suburban        Married  ...   
1      F       0      Suburban         Single  ...   
2      F   48767      Suburban        Married  ...   
3      M       0      Suburban        Married  ...   
4      M   43836         Rural         Single  ...   

   months_since_policy_inception  number_of_open_complaints  \
0                              5                          0   
1                             42                        

In [15]:
# 6. Save to a temporary clean CSV (or keep in memory for the next step)
df.to_csv('insurance_cleaned.csv', index=False)
print("\nSuccess: Cleaned data saved to 'insurance_cleaned.csv'")


Success: Cleaned data saved to 'insurance_cleaned.csv'


In [ ]:
import urllib.parse
from sqlalchemy import create_engine, text

USER = 'root'
PASSWORD = urllib.parse.quote_plus('<password>')
HOST = 'localhost'
PORT = '3306'
DB_NAME = 'insurance_db'

# Step 1: connect WITHOUT database
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/")

# Step 2: create DB if not exists
with engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))

# Step 3: connect TO the database
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")

# Step 4: load data
df.to_sql(
    name='customer_interactions',
    con=engine,
    if_exists='replace',
    index=False
)

print("Success: Data loaded!")

Success: Data loaded!
